# 07 - Build the modelling tables
**Input:** everything in `data/interim/` (run notebooks 01-04 first)
**Output:** `data/processed/demand_zone_30min.csv.gz` (forecaster), `data/processed/incidents_sim.csv.gz` (simulator),
`data/processed/service_times.csv` (empirical service-time distributions)

Steps: map each incident to a zone -> check coverage -> aggregate to zone x 30-min buckets (zero-filled) ->
attach weather and calendar features.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "README.md").exists())
sys.path.insert(0, str(ROOT / "src"))
RAW, INTERIM, PROCESSED = ROOT / "data/raw", ROOT / "data/interim", ROOT / "data/processed"
NB_DATA = ROOT / "notebooks/data"
INTERIM.mkdir(parents=True, exist_ok=True); PROCESSED.mkdir(parents=True, exist_ok=True)
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 40)

In [ ]:
inc = pd.read_csv(INTERIM / "ems_incidents_clean.csv.gz", parse_dates=["incident_datetime"], dtype={"zipcode": str})
xw = pd.read_csv(INTERIM / "zip_to_modzcta.csv", dtype=str)
zones = pd.read_csv(INTERIM / "zones_modzcta.csv", dtype={"zone": str})
inc = inc.merge(xw, on="zipcode", how="left")
print(f"{len(inc):,} incidents")
print(f"ZIP present: {inc.zipcode.notna().mean():.1%} | mapped to a MODZCTA zone: {inc.zone.notna().mean():.1%}")
print(f"community district present: {inc.communitydistrict.notna().mean():.1%}")
print(inc[inc.zone.isna()].zipcode.value_counts(dropna=False).head(8))

### Decision gate: zone unit
If MODZCTA coverage above is >= ~95%, use MODZCTA (finer, joins to Census). Otherwise fall back to community
district. Record the decision in `docs/data.md`.

In [ ]:
inc = inc.dropna(subset=["zone"]).copy()
inc["bucket"] = inc.incident_datetime.dt.floor("30min")
counts = inc.groupby(["zone", "bucket"]).size().rename("n_incidents")
low_mid = inc[inc.acuity.isin(["mid(4-6)", "low(7-8)"])].groupby(["zone", "bucket"]).size().rename("n_low_mid")

# zero-filled full grid so the model sees quiet periods too
idx = pd.MultiIndex.from_product([sorted(inc.zone.unique()), pd.date_range(inc.bucket.min(), inc.bucket.max(), freq="30min")], names=["zone", "bucket"])
d = pd.concat([counts, low_mid], axis=1).reindex(idx).fillna(0).astype("int16").reset_index()
print(d.shape, "| mean incidents per zone-bucket:", d.n_incidents.mean().round(3))

In [ ]:
w = pd.read_csv(INTERIM / "weather_hourly.csv", parse_dates=["time"])
cal = pd.read_csv(INTERIM / "calendar_daily.csv", parse_dates=["date"])
d["hour_ts"] = d.bucket.dt.floor("h"); d["date"] = d.bucket.dt.floor("D")
d = d.merge(w.rename(columns={"time": "hour_ts"}), on="hour_ts", how="left")
d = d.merge(cal[["date", "is_holiday", "is_heat_day", "is_heavy_rain"]], on="date", how="left")
d["hour"] = d.bucket.dt.hour + d.bucket.dt.minute / 60
d["dow"] = d.bucket.dt.dayofweek; d["month"] = d.bucket.dt.month; d["year"] = d.bucket.dt.year
d = d.drop(columns=["hour_ts", "date"])
print("rows without weather:", int(d.temperature_2m.isna().sum()))
d.to_csv(PROCESSED / "demand_zone_30min.csv.gz", index=False)

In [ ]:
# demand is strongly time-structured -> good for the baseline (zone x weekday x hour)
tot = d.groupby("bucket").n_incidents.sum()
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
tot.groupby([tot.index.dayofweek, tot.index.hour]).mean().unstack().T.plot(ax=ax[0], title="Mean citywide incidents per 30 min, by hour & weekday")
d.groupby("zone").n_incidents.sum().sort_values().tail(20).plot.barh(ax=ax[1], title="Top 20 zones by volume"); plt.tight_layout(); plt.show()

## Simulator inputs: incidents and empirical service times

In [ ]:
sim = inc[["incident_id", "incident_datetime", "zone", "borough", "acuity", "final_severity_level_code",
           "response_s", "travel_s", "onscene_s", "to_hosp_s", "job_s", "transported"]]
sim.to_csv(PROCESSED / "incidents_sim.csv.gz", index=False)

svc = sim.groupby("acuity", observed=True)[["onscene_s", "to_hosp_s", "job_s"]].describe(percentiles=[.5, .9]).T.round(0)
svc.to_csv(PROCESSED / "service_times.csv"); svc